# IPI Detection — Gemma-1.1-2B + LoRA Inference

Loads the **base** `google/gemma-1.1-2b-it` model and merges the fine-tuned LoRA adapter  
from a saved checkpoint, then runs inference on the held-out IPI test set.

**Step 1:** Run Cell 1 (environment setup) and **restart the kernel**.  
**Step 2:** Edit the paths in Cell 2 (config), then run all cells.

## 1 · Environment setup

> **Run this cell, restart the kernel, then proceed.**  
> Fixes `ModuleNotFoundError: No module named 'numpy.strings'`.
> Scipy/sklearn in the Kaggle image were built against NumPy ≥ 2.0; this upgrades NumPy to match.

In [ ]:
import subprocess, sys

pkgs = ["numpy>=2.0", "scipy", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs)
print("\n✅ Done. Restart the kernel now, then run all remaining cells.")

## 2 · Config

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
BASE_MODEL_ID   = "google/gemma-1.1-2b-it"
CHECKPOINT_PATH = "/kaggle/input/models/itbaansawan/epoch2-gemini-1-1-2b-isec-exp-1/transformers/default/1/checkpoint-3192"
TEST_JSON_PATH  = "/kaggle/input/datasets/ipi/test_dataset_ipi.json"  # adjust to your mount
# ────────────────────────────────────────────────────────────────────────────

MAX_NEW_TOKENS  = 512
BATCH_SIZE      = 4       # lower if OOM
NUM_SAMPLES     = None    # None → run full test set; set an int to limit (e.g. 50)

## 3 · Imports & device

In [ ]:
import json, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 4 · Load tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for batch generation
print(f"Vocab size : {tokenizer.vocab_size}")
print(f"pad_token  : {tokenizer.pad_token!r}  (id={tokenizer.pad_token_id})")

## 5 · Load base model (4-bit) + LoRA adapter

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_storage=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH)
model.eval()
print("Model loaded ✓")

## 6 · Load the test set

In [ ]:
with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

if NUM_SAMPLES is not None:
    test_data = test_data[:NUM_SAMPLES]

print(f"Test samples: {len(test_data)}")
print("Sample types:", {d['sample_type'] for d in test_data})

## 7 · Prompt builder

Reconstructs the **user-turn only** (no model turn) from the stored `system` and `user` fields,  
using the same Gemma chat template the training script used.

In [ ]:
GEMMA_USER_START  = "<start_of_turn>user\n"
GEMMA_USER_END    = "<end_of_turn>\n"
GEMMA_MODEL_START = "<start_of_turn>model\n"

def build_prompt(sample: dict) -> str:
    """Build the inference prompt (no model answer appended)."""
    system = sample["system"]
    user   = sample["user"]
    return (
        GEMMA_USER_START
        + system + "\n\n" + user
        + GEMMA_USER_END
        + GEMMA_MODEL_START   # model continues from here
    )

# Sanity-check on the first sample
print(build_prompt(test_data[0])[:500])

## 8 · Batch generation helper

In [ ]:
def generate_batch(prompts: list) -> list:
    """Tokenize a batch of prompts, generate, and decode the new tokens only."""
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,          # greedy — deterministic
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = [
        tokenizer.decode(out[input_len:], skip_special_tokens=True)
        for out in outputs
    ]
    return decoded

## 9 · Run inference over the test set

In [ ]:
from tqdm.auto import tqdm

results = []

for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc="Generating"):
    batch = test_data[i : i + BATCH_SIZE]
    prompts = [build_prompt(s) for s in batch]
    preds   = generate_batch(prompts)

    for sample, pred in zip(batch, preds):
        results.append({
            "sample_type":   sample["sample_type"],
            "user":          sample["user"],
            "expected":      sample["assistant"],
            "predicted":     pred,
        })

print(f"Done. {len(results)} predictions generated.")

## 10 · Parse & evaluate predictions

Both `expected` and `predicted` are supposed to be JSON strings.  
We try to parse them and compare field by field.

In [ ]:
def try_parse_json(text: str):
    """Attempt to parse JSON; fall back to None on failure."""
    text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.MULTILINE)
    text = re.sub(r"```$",             "", text.strip(), flags=re.MULTILINE)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def eval_prediction(expected_str: str, predicted_str: str) -> dict:
    exp  = try_parse_json(expected_str)
    pred = try_parse_json(predicted_str)

    if exp is None or pred is None:
        return {"parse_ok": False, "fake_ok": False, "real_ok": False, "answer_ok": False}

    def norm(v):
        if isinstance(v, list): return json.dumps(v, ensure_ascii=False, sort_keys=True)
        return str(v).strip() if v is not None else ""

    fake_ok   = norm(exp.get("fake_instruction"))    == norm(pred.get("fake_instruction"))
    real_ok   = norm(exp.get("real_instruction"))    == norm(pred.get("real_instruction"))
    answer_ok = norm(exp.get("final_answer_to_task")) == norm(pred.get("final_answer_to_task"))

    return {"parse_ok": True, "fake_ok": fake_ok, "real_ok": real_ok, "answer_ok": answer_ok}


for r in results:
    r["eval"] = eval_prediction(r["expected"], r["predicted"])

print("Evaluation attached to results ✓")

## 11 · Summary metrics

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"sample_type": r["sample_type"], **r["eval"]}
    for r in results
])

overall = df[["parse_ok", "fake_ok", "real_ok", "answer_ok"]].mean()
print("=== Overall accuracy ===")
print(overall.to_string())

print("\n=== By sample_type ===")
print(df.groupby("sample_type")[["parse_ok","fake_ok","real_ok","answer_ok"]].mean().to_string())

## 12 · Inspect individual predictions

In [ ]:
def show(idx: int):
    r = results[idx]
    print(f"[{idx}] type={r['sample_type']}")
    print()
    print("── USER PROMPT (truncated 600 chars) ──")
    print(r["user"][:600])
    print()
    print("── EXPECTED ──")
    try:
        print(json.dumps(json.loads(r["expected"]), indent=2, ensure_ascii=False))
    except Exception:
        print(r["expected"])
    print()
    print("── PREDICTED ──")
    pred_parsed = try_parse_json(r["predicted"])
    if pred_parsed:
        print(json.dumps(pred_parsed, indent=2, ensure_ascii=False))
    else:
        print(r["predicted"])
    print()
    print("── EVAL ──", r["eval"])

for i in range(min(3, len(results))):
    show(i)
    print("=" * 80)

## 13 · Save results to JSON

In [ ]:
OUT_PATH = "/kaggle/working/ipi_inference_results.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Results saved → {OUT_PATH}")